In [ ]:
#pip install scgraph==3.2.2

In [ ]:
#pip install osmnx

In [ ]:
# pip install folium

In [1]:
import osmnx as ox
import folium
from scgraph import GeoGraph

In [2]:
# Pull the bike network for Somerville and Cambridge, MA
G = ox.graph_from_place(
    ['Somerville, Massachusetts, USA', 'Cambridge, Massachusetts, USA'],
    network_type='bike'
)

In [3]:
# Add bike speeds and compute travel times (in seconds) for each edge
G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)

In [4]:
# Load time-based geograph (edge weights in seconds)
geograph_time = GeoGraph.load_from_osmnx_graph(G, weight_key='travel_time')

# Load distance-based geograph (edge weights in kilometers)
geograph_distance = GeoGraph.load_from_osmnx_graph(G, weight_key='length')

No off_graph_travel_speed provided but weight_key = 'travel_time'. Assuming instant travel off graph. Setting default_off_graph_circuity to 0 and default_node_addition_circuity to 240.


In [5]:
# MIT campus -> Somerville City Hall
origin = {'latitude': 42.3601, 'longitude': -71.0942}
destination = {'latitude': 42.3876, 'longitude': -71.0995}

# Shortest path by biking time (output_path=True exposes the on-graph node id list)
time_result = geograph_time.get_shortest_path(
    origin_node=origin, destination_node=destination, output_path=True
)

# Shortest path by distance (output_path=True for the same reason)
distance_result = geograph_distance.get_shortest_path(
    origin_node=origin, destination_node=destination, output_path=True
)

# Cross-evaluate: distance of the time-optimal path, and time of the distance-optimal path
time_path_distance_km = geograph_distance.get_path_weight(time_result)
distance_path_time_s = geograph_time.get_path_weight(distance_result)

print(f"Time-optimal path:     {time_result['length']:.1f} s  |  {time_path_distance_km:.3f} km")
print(f"Distance-optimal path: {distance_path_time_s:.1f} s  |  {distance_result['length']:.3f} km")

Time-optimal path:     340.9 s  |  3.920 km
Distance-optimal path: 369.3 s  |  3.605 km


In [6]:
# Plot both paths on a folium map
map_center = [
    (origin['latitude'] + destination['latitude']) / 2,
    (origin['longitude'] + destination['longitude']) / 2,
]
m = folium.Map(map_center, zoom_start=14)

# Time-optimal path (blue)
folium.PolyLine(
    time_result['coordinate_path'],
    color='blue',
    weight=5,
    opacity=0.7,
    popup=f"Time-optimal: {time_result['length']:.1f} s, {time_path_distance_km:.3f} km",
).add_to(m)

# Distance-optimal path (green)
folium.PolyLine(
    distance_result['coordinate_path'],
    color='green',
    weight=5,
    opacity=0.7,
    popup=f"Distance-optimal: {distance_result['length']:.3f} km, {distance_path_time_s:.1f} s",
).add_to(m)

# Origin and destination markers
folium.Marker([origin['latitude'], origin['longitude']], popup='Origin (MIT campus)').add_to(m)
folium.Marker([destination['latitude'], destination['longitude']], popup='Destination (Somerville City Hall)').add_to(m)

# Legend
legend_html = '''
     <div style="position: fixed;
                 bottom: 50px; left: 50px;
                 padding: 10px;
                 border:2px solid grey; z-index:9999; font-size:14px;
                 background-color:white;">
         <h3>Legend</h3>
         <i class="fa fa-circle" style="color:blue"></i>&nbsp;Time-optimal path<br>
         <i class="fa fa-circle" style="color:green"></i>&nbsp;Distance-optimal path
      </div>
'''
m.get_root().html.add_child(folium.Element(legend_html))
m